# First-order properties

First-order properties quantify expectation values of one-electron operators in individual electronic states. For the electronic ground state, quantities such as the permanent electric dipole moment are evaluated directly from the converged Kohn–Sham density, requiring no response treatment. In contrast, expectation values for electronically excited states—most notably excited-state dipole moments—are obtained from the double residue of the quadratic response function, which provides the transition-specific way to construct expectation values in the excited-state manifold. Together, these formulations provide a consistent route to state-specific first-order properties across both ground and excited electronic states. See {cite}`Norman2018` for further details.

## Electric dipole moment

### Ground state



In [69]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("para-nitroaniline")
#molecule = vlx.Molecule.read_name("water")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
#scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(molecule, basis)

prop_drv = vlx.FirstOrderPropertyDriver()

prop_drv.property = "electric dipole moment"

prop_results = prop_drv.compute(molecule, basis, scf_results)

print(prop_results)

Reading para-nitroaniline from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Dir

In [70]:
import numpy as np

results = prop_results["electric dipole moment"]

print("Electric dipole moment (a.u.):")
print(f"{"x":>17s} {"y":>8s} {"z":>8s}")

print(
    f"   nuclear: {results["nuclear"][0]:>8.4f} {results["nuclear"][1]:>8.4f} {results["nuclear"][2]:>8.4f}"
)
print(
    f"electronic: {results["electronic"][0]:>8.4f} {results["electronic"][1]:>8.4f} {results["electronic"][2]:>8.4f}"
)
print(
    f"     total: {results["total"][0]:>8.4f} {results["total"][1]:>8.4f} {results["total"][2]:>8.4f}"
)

dipmom = np.linalg.norm(results["total"])
au2debye = 2.5417464

print(f"\nMagnitude: {dipmom * au2debye: .4f} Debye")

Electric dipole moment (a.u.):
                x        y        z
   nuclear:  -0.0000  -0.0000   0.0000
electronic:  -2.6675  -1.4204   1.4941
     total:  -2.6675  -1.4204   1.4941

Magnitude:  8.5688 Debye


In [68]:
molecule.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Excited state

**Python script**

In [38]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("water")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "cam-b3lyp"
scf_results = scf_drv.compute(molecule, basis)

Reading water from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Kohn-Sham                                            
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversio

In [71]:
from veloxchem.doubleresbeta import DoubleResBetaDriver

In [72]:
excmom_drv = DoubleResBetaDriver()

excmom_drv.initial_state = 1
excmom_drv.final_state = 1

excmom_results = excmom_drv.compute(molecule, basis, scf_results)

                                                                                                                          
                                             Quadratic Response Driver Setup                                              
                                                                                                                          
                                    ERI Screening Threshold         : 1.0e-12                                             
                                    Convergance Threshold           : 1.0e-04                                             
                                    Max. Number of Iterations       : 150                                                 
                                    Max. Number of Iterations       : 150                                                 
                                                                                                                          
                

In [73]:
excmom_results

{'photon_energies': [np.float64(-0.0)],
 'ground_state_dipole_moments': array([-2.66750898, -1.42036417,  1.49407172]),
 'excited_state_dipole_moments': {('x', 1, 1): np.float64(-1.545533813384877),
  ('y', 1, 1): np.float64(-0.5651971114485023),
  ('z', 1, 1): np.float64(0.6993768502933555)},
 'oscillator_strengths': array([0.45185299]),
 'elec_trans_dipoles': array([[-1.47781056, -0.89126293,  0.89505616]]),
 'excitation_details': [['HOMO     -> LUMO        -0.9270',
   'HOMO-4   -> LUMO        -0.2172',
   'HOMO-1   -> LUMO+1      -0.2130']]}

**Text file**

:::{code}
jobs
task: response
@end

@method settings
xcfun: cam-b3lyp
basis: def2-svpd
@end

@response
property: transition dipole moment
initial_state: 1
final_state: 1
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
:::
